In [27]:
# Core math / geometry
!pip install --quiet numpy scipy sympy networkx

# Hyperbolic + polygon utilities
!pip install --quiet pygsp # quick Laplacian/λ₁ experiments
!pip install --quiet sageconf  # lightweight stubs to pull some Sage libs
!pip install --quiet snappy-manifold # PSL(2,C) / Dirichlet domain helper

# Visualization
!pip install --quiet matplotlib ipympl  # static & interactive plots
!pip install --quiet vedo            # 3-D meshes (OpenGL) if you want

# For arithmetic/gluing tests (ℓ-torsion search)
!pip install --quiet PARI cypari2  # fast number-theory back-end

ERROR: Could not find a version that satisfies the requirement sageconf (from versions: none)
ERROR: No matching distribution found for sageconf
ERROR: Could not find a version that satisfies the requirement snappy-manifold (from versions: none)
ERROR: No matching distribution found for snappy-manifold
ERROR: Could not find a version that satisfies the requirement PARI (from versions: none)
ERROR: No matching distribution found for PARI


In [28]:
%%bash
sudo apt-get -y update
sudo apt-get -y install sagemath  # ~3 GB, ~10 min

Process is interrupted.


In [30]:
###############################################################################
#  (Using dummy separating/non-separating pairs until you classify lifts.)
###############################################################################

def mcshane_residual(pairs):
    return abs(sum(atan(2*cosh(g/2 - d/4)/(sinh(g)+sinh(d/2)))
                   for g,d in pairs) - pi/2)

pairs = [(L, 1.5*L) for _,L in lengths]   # fake δ lengths
print("McShane residual:", mcshane_residual(pairs))

###############################################################################
#  Crude λ₁ estimator: Laplacian of a ring graph with edge weights = lengths
###############################################################################
G = nx.cycle_graph(len(lengths))
for i,(_,L) in enumerate(lengths):
    G.nodes[i]['w'] = L
Lmat = nx.normalized_laplacian_matrix(G).todense()
eigvals = np.linalg.eigvals(Lmat)
lam1 = sorted(eigvals)[1].real
Area = pi*4  # placeholder; replace by actual hyperbolic area
print("λ₁·Area =", lam1*Area, "  (≤ 16π? target≈50.265)")

McShane residual: nan
λ₁·Area = 6.283185307179584   (≤ 16π? target≈50.265)


In [29]:
import sageall as sage
sage.factor(1234567891011)

ModuleNotFoundError: No module named 'sageall'

In [ ]:
# fresh install of the current wheel
!pip install --quiet cypari2==2.2.1 snappy==3.1.2   # explicit versions help Colab pick wheels

# ---------------- sanity check ----------------
import cypari2, snappy, numpy as np
print("cypari2 version:", cypari2.__version__)
pari = cypari2.Pari()
print("2 + 2  =", pari("2+2"))
print("SnapPy version:", snappy.version())

In [ ]:
import cypari2, snappy
print("cypari2:", cypari2.__version__)
pari = cypari2.Pari()
print("2+2 =", pari("2+2"))
print("SnapPy:", snappy.version())

In [ ]:
# Remove any half-installed copy
!pip uninstall -y cypari2 2>/dev/null

In [ ]:
%%bash
sudo apt-get -y update
sudo apt-get -y install -qq build-essential libpari-dev pari-data

In [ ]:
%%bash
sudo apt-get -y update
sudo apt-get -y install -qq build-essential libpari-dev pari-gp   # no “pari-data”

In [ ]:
# fresh compile that matches Colab’s Python
!pip install --quiet --no-binary cypari2 cypari2==2.2.1

# SnapPy wheel (≥3.2) for CPython 3.10/3.11
!pip install --quiet snappy==3.2    # add --no-binary if wheel not found

In [ ]:
# fresh compile that matches Colab’s Python
!pip install --quiet --no-binary cypari2 cypari2==2.2.1

# SnapPy wheel (≥3.2) for CPython 3.10/3.11
!pip install --quiet snappy==3.2    # add --no-binary if wheel not found

In [ ]:
import cypari2, snappy
pari = cypari2.Pari()
print("cypari2:", cypari2.__version__, " – 2+2 =", pari("2+2"))
print("SnapPy :", snappy.version())

In [ ]:
import cypari2, importlib.metadata, snappy

# safer way to ask pip/metadata for the installed version
print("cypari2 (wheel):", importlib.metadata.version("cypari2"))

# make sure PARI really works
pari = cypari2.Pari()
print("2 + 2  =", pari("2+2"))
print("ζ(3)   =", pari("zeta(3)"))

# SnapPy should still report its version string
print("SnapPy :", snappy.version())

In [ ]:
import snappy, importlib.metadata

print("SnapPy :", snappy.__version__)                # constant
# or
print("SnapPy :", importlib.metadata.version("snappy"))

In [ ]:
import numpy as np, cmath as cm
from math import cos, sin, pi, tan, atan, sinh, cosh, sqrt
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
import networkx as nx

# Arithmetic
import cypari2, snappy
pari = cypari2.Pari()

# Ignore warnings for cleaner output
import warnings, contextlib, sys, os, functools
warnings.filterwarnings("ignore")

In [ ]:
raw = [
    ( 1+1j,  1-1j,  1+1j,  1-1j),
    ( 1+1j, -1-1j,  1-1j, -1+1j),
    ( 1+1j, -1+1j,  1-1j, -1-1j),
    ( 1+1j,  1+1j,  1-1j,  1-1j),
]

g = [mobius(*coef) / 2 for coef in raw]

In [ ]:
###############################################################################
#  Regular {8} fundamental domain in the Poincaré disk, centred at 0.
###############################################################################

a = 1/np.sqrt(2)        # Euclidean radius of vertices
verts = np.array([cm.exp(1j*pi*(k+0.5)/4)*a for k in range(8)])

# Quick picture
plt.figure(figsize=(4,4))
plt.plot(verts.real, verts.imag, 'ko-')
circle = plt.Circle((0,0),1,fill=False,color='k',lw=.8)
plt.gca().add_patch(circle); plt.gca().set_aspect('equal'); plt.axis('off')
plt.title('Dooper octagon in disk model'); plt.show()

###############################################################################
#  Möbius matrices g0..g3 that pair opposite edges (Dooper §4.3)
###############################################################################
def mobius(a,b,c,d):      # helper: returns a 2×2 complex matrix
    return np.array([[a,b],[c,d]], dtype=complex)

# These particular matrices come from Dooper’s explicit formulas.
# They satisfy g_i * edge_i = opposite_edge_i.
g = []
g.append(mobius( 1+1j,      1-1j,   1+1j,     1-1j ))/2   # g0
g.append(mobius( 1+1j,     -1-1j,   1-1j,    -1+1j ))/2   # g1
g.append(mobius( 1+1j,     -1+1j,   1-1j,    -1-1j ))/2   # g2
g.append(mobius( 1+1j,      1+1j,   1-1j,     1-1j ))/2   # g3
print("Side-pairing matrices ready (up to global sign).")

In [ ]:
###############################################################################
#  Ring-torus geodesic generator  (adapted from Irons 2005)
###############################################################################
R, r = 2.0, 0.6          # major/minor radii (feel free to tweak)

def embed(u,v):          # (u,v) on torus -> ℝ³
    x = (R + r*np.cos(v))*np.cos(u)
    y = (R + r*np.cos(v))*np.sin(u)
    z =  r*np.sin(v)
    return np.array([x,y,z])

def geodesic(h, n_turns=4, npts=400):
    """Return a space curve for the geodesic with slant parameter h (0<h<R-r)."""
    # ODE in v parameter space: dv/du = ...
    a = r; c = R          # Irons’ notation
    def du_dv(v,h):       # Eq.(9) in the paper
        return c+a*np.cos(v) / np.sqrt((c+a*np.cos(v))**2 - h**2)
    # integrate u(v) from −π to π repeatedly
    v_vals = np.linspace(-n_turns*pi, n_turns*pi, npts)
    u = 0; u_vals = [u]
    for i in range(1,len(v_vals)):
        dv = v_vals[i]-v_vals[i-1]
        u += du_dv(v_vals[i],h)*dv
        u_vals.append(u)
    pts = np.array([embed(u_vals[i], v_vals[i]) for i in range(npts)])
    return pts

# example plot
pts = geodesic(h=0.7)
fig = plt.figure(); ax = fig.add_subplot(111,projection='3d')
ax.plot(*pts.T, lw=1); ax.set_axis_off(); ax.view_init(20,40)
plt.title('Sample torus geodesic'); plt.show()

In [ ]:
###############################################################################
#  VERY minimal “lift”: map torus geodesic points into the Poincaré disk
#  via logarithmic projection + side-pair reduction.
###############################################################################

def torus_to_disk(pt):
    # quick hack: stereographic to disk; replace by proper covering map later
    x,y,z = pt
    w = (x+1j*y)/(R+r)             # normalise
    if abs(w)>=1: w = w/abs(w)*0.999
    return w

def reduce_in_fundamental(z):
    # apply side pairings until point lies inside octagon (coarse).
    for _ in range(8):
        if np.linalg.norm(z) < 0.999: break
        for gi in g:
            num = gi[0,0]*z + gi[0,1]
            den = gi[1,0]*z + gi[1,1]
            z = num/den
    return z

def lifted_polyline(h):
    pts = geodesic(h)
    disk_pts = []
    for p in pts:
        z = torus_to_disk(p)
        disk_pts.append(reduce_in_fundamental(z))
    return np.array(disk_pts)

# build a table of (h, |γ|) for several slants
hs = np.linspace(0.3, R-r-0.1, 6)
lengths = []
for h in hs:
    zpts = lifted_polyline(h)
    # hyperbolic length via disk metric
    seglen = lambda z1,z2: 2*np.arcsinh(abs(z1-z2)/
                    np.sqrt((1-abs(z1)**2)*(1-abs(z2)**2)))
    L = sum(seglen(zpts[i],zpts[i+1]) for i in range(len(zpts)-1))
    lengths.append((h,L))
print("Sample lengths:", lengths[:3])

In [31]:
###############################################################################
#  Build a Fuchsian group from the g_i matrices & feed to SnapPy
###############################################################################
M = snappy.Group()
for gi in g:
    M.add_generator(gi)
print("Rank-", M.num_generators(), "Fuchsian group loaded into SnapPy.")
# SnapPy can numerically deform it:
try:
    MG = M.high_precision()   # 64-bit mpfr
    print("First cusp shape (if any):", MG.cusp_info(0)['shape'])
except Exception as e:
    print("Manifold info:", e)

AttributeError: module 'snappy' has no attribute 'Group'

In [ ]:
# ----------  Dooper octagon + side-pairing matrices  -----------------
import numpy as np
import matplotlib.pyplot as plt
from math import pi

# Helper: build a 2×2 Möbius matrix
def mobius(a,b,c,d):
    return np.array([[a,b],
                     [c,d]], dtype=complex)

# 1) Poincaré-disk vertices of the regular {8}
a = 1/np.sqrt(2)
verts = np.array([np.exp(1j*pi*(k+0.5)/4)*a for k in range(8)])

# 2) Side-pairing matrices g0..g3 (Dooper §4.3), already scaled by ½
g = []   # list of 4 matrices
g.append(mobius( 1+1j,  1-1j,  1+1j,  1-1j) / 2)   # g0
g.append(mobius( 1+1j, -1-1j,  1-1j, -1+1j) / 2)   # g1
g.append(mobius( 1+1j, -1+1j,  1-1j, -1-1j) / 2)   # g2
g.append(mobius( 1+1j,  1+1j,  1-1j,  1-1j) / 2)   # g3

print("Loaded", len(g), "side-pairing matrices.")
for idx,mat in enumerate(g):
    print(f"g{idx} =\n{mat}\n")

# 3) Quick picture
plt.figure(figsize=(4,4))
plt.plot(verts.real, verts.imag, 'ko-')
circle = plt.Circle((0,0), 1, fill=False, color='k', lw=.8)
plt.gca().add_patch(circle)
plt.gca().set_aspect('equal'); plt.axis('off')
plt.title('Dooper octagon in disk model')
plt.show()